In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.utils import resample

# Load dữ liệu gốc từ các file Excel
# Chú ý: Nếu chạy file bằng Google colab thì chạy 2 dòng dưới để chỉ data vào đúng chỗ
from google.colab import drive
drive.mount('/content/gdrive')
# Chú ý: Thay đổi đường dẫn cho đúng với đường dẫn của máy tính
df_companies = pd.read_excel('/content/gdrive/MyDrive/DL07/Project02/data/Overview_Companies.xlsx')
df_reviews = pd.read_excel('/content/gdrive/MyDrive/DL07/Project02/data/Reviews.xlsx')

Mounted at /content/gdrive


In [ ]:
# ---------------------------------------------------------
# TASK 1: TIỀN XỬ LÝ DỮ LIỆU CÔNG TY (CONTENT-BASED)
# ---------------------------------------------------------

df_comp = df_companies.copy()

# 1. Gom các cột văn bản chính lại với nhau. Lấp đầy các ô trống bằng chuỗi rỗng để tránh lỗi NaN.
df_comp['Text_Features'] = (
    df_comp['Company overview'].fillna('') + ' ' +
    df_comp['Our key skills'].fillna('') + ' ' +
    df_comp["Why you'll love working here"].fillna('')
)

# 2. Tạo chuỗi metadata từ các đặc trưng phân loại (Company Type, Industry, Size).
# Việc thay thế khoảng trắng bằng dấu gạch dưới giúp TF-IDF coi "1-50_employees" là một từ khóa nguyên vẹn.
df_comp['Metadata'] = (
    df_comp['Company Type'].fillna('').str.replace(' ', '_') + ' ' +
    df_comp['Company industry'].fillna('').str.replace(' ', '_') + ' ' +
    df_comp['Company size'].fillna('').str.replace(' ', '_')
)

# 3. Trích xuất thành phố chính từ cột Location (dạng chuỗi dictionary phức tạp)
def extract_main_city(loc_str):
    if pd.isna(loc_str): return ""
    loc_str = str(loc_str)
    if 'Ho Chi Minh' in loc_str: return "ho_chi_minh"
    elif 'Ha Noi' in loc_str: return "ha_noi"
    elif 'Da Nang' in loc_str: return "da_nang"
    return "other_city"

df_comp['Main_City'] = df_comp['Location'].apply(extract_main_city)

# 4. Ghép toàn bộ lại thành một "Tài liệu" (Document) hoàn chỉnh cho mỗi công ty
df_comp['Final_Content'] = df_comp['Text_Features'] + ' ' + df_comp['Metadata'] + ' ' + df_comp['Main_City']

# 5. Làm sạch văn bản: chuẩn hóa chữ thường, xóa ký tự đặc biệt, số và stop-words
VIETNAMESE_STOP_WORDS = {
    "và", "của", "là", "có", "được", "cho", "với", "các", "những", "một",
    "trong", "trên", "khi", "này", "đó", "để", "từ", "như", "về", "không",
    "cũng", "đã", "sẽ", "bị", "vào", "ra", "nên", "hay", "hoặc", "rất",
    "nhiều", "hơn", "nhất", "tại", "ở", "thì", "mà", "còn", "nữa", "lại",
    "mình", "bạn", "họ", "chúng", "tôi", "anh", "chị", "em", "người",
    "việc", "công", "ty",  # "công"+"ty" đã tách; có thể bỏ nếu muốn giữ "côngty"
}

def clean_text(text):
    text = str(text).lower()
    words = re.findall(
        r"\b[a-zàáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳỵỷỹ]+\b",
        text,
    )
    stop_words = set(ENGLISH_STOP_WORDS) | VIETNAMESE_STOP_WORDS
    tokens = [w for w in words if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

df_comp['Cleaned_Content'] = df_comp['Final_Content'].apply(clean_text)

# Kiểm tra nhanh kết quả
print("Kích thước dữ liệu công ty sẵn sàng cho TF-IDF:", df_comp[['Company Name', 'Cleaned_Content']].shape)

Kích thước dữ liệu công ty sẵn sàng cho TF-IDF: (478, 2)


In [ ]:
# Xuất File 1
columns_to_keep_comp = ['id', 'Company Name', 'Cleaned_Content', 'Href']
df_comp[columns_to_keep_comp].to_csv('Cleaned_Companies_TFIDF.csv', index=False, encoding='utf-8-sig')

In [ ]:
# =========================================================
# YÊU CẦU 2: FEATURE ENGINEERING CHO BÀI TOÁN PHÂN LOẠI
# =========================================================
df_rev = df_reviews.copy()
df_rev = df_rev.dropna(subset=['Recommend?'])

# Tạo đặc trưng mới
sub_rating_cols = ['Salary & benefits', 'Training & learning', 'Management cares about me', 'Culture & fun', 'Office & workspace']
df_rev['Average_Sub_Rating'] = df_rev[sub_rating_cols].mean(axis=1)
df_rev['Rating_Discrepancy'] = df_rev['Rating'] - df_rev['Average_Sub_Rating']

df_rev['Liked_Length'] = df_rev['What I liked'].fillna('').apply(lambda x: len(str(x).split()))
df_rev['Improve_Length'] = df_rev['Suggestions for improvement'].fillna('').apply(
    lambda x: len(str(x).split())
)
df_rev['Review_Length'] = df_rev['Liked_Length'] + df_rev['Improve_Length']

# Mã hóa mục tiêu và Text
df_rev['Target'] = df_rev['Recommend?'].map({'Yes': 1, 'No': 0})
df_rev['Review_Text'] = df_rev['What I liked'].fillna('') + ' ' + df_rev['Suggestions for improvement'].fillna('')
df_rev['Cleaned_Review_Text'] = df_rev['Review_Text'].apply(clean_text)

features_to_keep = [
    'Rating', 'Salary & benefits', 'Training & learning',
    'Management cares about me', 'Culture & fun', 'Office & workspace',
    'Average_Sub_Rating', 'Rating_Discrepancy',
    'Liked_Length', 'Improve_Length', 'Review_Length',
    'Cleaned_Review_Text', 'Target'
]
df_ml_ready = df_rev[features_to_keep]

In [ ]:
# Xuất File 2 (Imbalanced)
df_imbalanced = df_ml_ready.copy()
df_imbalanced.to_csv('Reviews_Imbalanced_ML.csv', index=False, encoding='utf-8-sig')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# 1) Tách feature / target
X = df_imbalanced.drop(columns=['Target'])
y = df_imbalanced['Target']

# 2) Split TRƯỚC — giữ tỷ lệ thực tế trên test (~88/12)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3) Chỉ undersample TẬP TRAIN
train_df = X_train.copy()
train_df['Target'] = y_train.values

df_yes = train_df[train_df['Target'] == 1]
df_no  = train_df[train_df['Target'] == 0]

df_yes_downsampled = resample(
    df_yes,
    replace=False,
    n_samples=len(df_no),
    random_state=42
)

train_balanced = (
    pd.concat([df_yes_downsampled, df_no])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# 4) Test giữ nguyên imbalanced
test_imbalanced = X_test.copy()
test_imbalanced['Target'] = y_test.values

# 5) Export
# File full imbalanced (đã có từ cell trước) — dùng cho baseline
# Train balanced + Test imbalanced — dùng khi so sánh model
train_balanced.to_csv('Reviews_Train_Balanced.csv', index=False, encoding='utf-8-sig')
test_imbalanced.to_csv('Reviews_Test_Imbalanced.csv', index=False, encoding='utf-8-sig')

# (Tuỳ chọn) Giữ file cũ tên Reviews_Balanced_ML.csv = train balanced
# để không gãy notebook sau nếu đã hard-code tên file
train_balanced.to_csv('Reviews_Balanced_ML.csv', index=False, encoding='utf-8-sig')

print("Train balanced:", train_balanced['Target'].value_counts().to_dict())
print("Test imbalanced:", test_imbalanced['Target'].value_counts().to_dict())
print("Hoàn tất export train balanced + test imbalanced.")

Train balanced: {1: 826, 0: 826}
Test imbalanced: {1: 1478, 0: 206}
Hoàn tất export train balanced + test imbalanced.
